Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_lstm_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 6)
Las dimensiones de testX son:  (10533, 12, 6)
Las dimensiones de valX son:  (5189, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 13s - 58ms/step - ia: 0.5215 - loss: 0.4662 - mae: 0.5332 - rmse: 0.6722 - smape: 1.0684 - val_ia: 0.3471 - val_loss: 0.3797 - val_mae: 0.5323 - val_rmse: 0.5920 - val_smape: 0.9868

Epoch 2/128                                           

231/231 - 4s - 17ms/step - ia: 0.6744 - loss: 0.2960 - mae: 0.4221 - rmse: 0.5372 - smape: 0.8175 - val_ia: 0.4110 - val_loss: 0.2037 - val_mae: 0.3872 - val_rmse: 0.4320 - val_smape: 0.8536

Epoch 3/128                                           

231/231 - 4s - 19ms/step - ia: 0.7218 - loss: 0.2362 - mae: 0.3697 - rmse: 0.4808 - smape: 0.7228 - val_ia: 0.4739 - val_loss: 0.1306 - val_mae: 0.2917 - val_rmse: 0.3364 - val_smape: 0.7113

Epoch 4/128                                           

231/231 - 4s - 17ms/step - ia: 0.7407 - loss: 0.2119 - mae: 0.3499 - rmse: 0.4536 - smape: 0.6839 - val_ia: 0.4811 - val_loss: 0.1239 - val_mae: 0.2870 - val_rmse: 0.3272 - val_smape: 0.6937

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

29/29 - 15s - 530ms/step - ia: 0.2483 - loss: 0.6435 - mae: 0.6543 - rmse: 0.7931 - smape: 1.5177 - val_ia: 0.5599 - val_loss: 0.5343 - val_mae: 0.6322 - val_rmse: 0.7250 - val_smape: 1.0422

Epoch 2/16                                                                           

29/29 - 5s - 177ms/step - ia: 0.7376 - loss: 0.2288 - mae: 0.3644 - rmse: 0.4753 - smape: 0.7347 - val_ia: 0.7024 - val_loss: 0.1733 - val_mae: 0.3484 - val_rmse: 0.4150 - val_smape: 0.8408

Epoch 3/16                                                                           

29/29 - 5s - 180ms/step - ia: 0.8318 - loss: 0.1141 - mae: 0.2405 - rmse: 0.3333 - smape: 0.5166 - val_ia: 0.8479 - val_loss: 0.0762 - val_mae: 0.1976 - val_rmse: 0.2744 - val_smape: 0.5110

Epoch 4/16                                                                           

29/29 - 5s - 182ms/step - ia: 0.8805 - loss: 0.0601 - mae: 0.1745 - rmse: 0.24

...

Epoch 1/8                                                                              

116/116 - 8s - 68ms/step - ia: 0.8729 - loss: 0.0699 - mae: 0.1824 - rmse: 0.2376 - smape: 0.4120 - val_ia: 0.8985 - val_loss: 0.0129 - val_mae: 0.0872 - val_rmse: 0.1117 - val_smape: 0.2057

Epoch 2/8                                                                              

116/116 - 1s - 11ms/step - ia: 0.9251 - loss: 0.0217 - mae: 0.1097 - rmse: 0.1456 - smape: 0.2642 - val_ia: 0.9289 - val_loss: 0.0071 - val_mae: 0.0640 - val_rmse: 0.0822 - val_smape: 0.1785

Epoch 3/8                                                                              

116/116 - 1s - 12ms/step - ia: 0.9330 - loss: 0.0175 - mae: 0.0979 - rmse: 0.1309 - smape: 0.2389 - val_ia: 0.9482 - val_loss: 0.0046 - val_mae: 0.0471 - val_rmse: 0.0647 - val_smape: 0.1297

Epoch 4/8                                                                              

116/116 - 3s - 22ms/step - ia: 0.9365 - loss: 0.0162 - mae: 0.0931 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                             

922/922 - 30s - 32ms/step - ia: 0.8451 - loss: 0.0783 - mae: 0.1906 - rmse: 0.2422 - smape: 0.4435 - val_ia: 0.4932 - val_loss: 0.0216 - val_mae: 0.1030 - val_rmse: 0.1173 - val_smape: 0.2934

Epoch 2/16                                                                             

922/922 - 16s - 17ms/step - ia: 0.9152 - loss: 0.0226 - mae: 0.1086 - rmse: 0.1405 - smape: 0.2832 - val_ia: 0.4919 - val_loss: 0.0176 - val_mae: 0.0995 - val_rmse: 0.1123 - val_smape: 0.2609

Epoch 3/16                                                                             

922/922 - 16s - 18ms/step - ia: 0.9259 - loss: 0.0173 - mae: 0.0953 - rmse: 0.1232 - smape: 0.2568 - val_ia: 0.4269 - val_loss: 0.0227 - val_mae: 0.1193 - val_rmse: 0.1313 - val_smape: 0.2802

Epoch 4/16                                                                             

922/922 - 15s - 16ms/step - ia: 0.9336 - loss: 0.0143 - mae: 0.08

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                             

58/58 - 6s - 106ms/step - ia: 0.2636 - loss: 0.7610 - mae: 0.7178 - rmse: 0.8709 - smape: 1.4829 - val_ia: 0.3651 - val_loss: 0.8490 - val_mae: 0.7746 - val_rmse: 0.9112 - val_smape: 1.6805

Epoch 2/32                                                                             

58/58 - 0s - 8ms/step - ia: 0.2977 - loss: 0.7094 - mae: 0.6913 - rmse: 0.8421 - smape: 1.4267 - val_ia: 0.3891 - val_loss: 0.8092 - val_mae: 0.7565 - val_rmse: 0.8891 - val_smape: 1.6122

Epoch 3/32                                                                             

58/58 - 0s - 8ms/step - ia: 0.3404 - loss: 0.6431 - mae: 0.6587 - rmse: 0.8007 - smape: 1.3683 - val_ia: 0.4103 - val_loss: 0.7562 - val_mae: 0.7307 - val_rmse: 0.8591 - val_smape: 1.5254

Epoch 4/32                                                                             

58/58 - 0s - 7ms/step - ia: 0.3887 - loss: 0.5828 - mae: 0.6235 - rmse: 0.7

In [16]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.0, 'epochs': 5, 'layers': 2.0, 'learning_rate': 0.0004934777489968929, 'units': 4}
